# Manual backtest notebook — v22 SNIPER recipe on XAUUSD

This notebook drives the **real engine** through the repo's own canonical pipeline.
It does not re-implement the strategy: every knob comes from `bt/optimal_config.py`,
every artifact from `runtime.sweeps.cached_sweep.ArtifactCache`, and every verdict from
`runtime/diagnostics/leg_buckets.py`.

> **The unit of evaluation is the LEG, not the month.** A month is a *reporting*
> dimension, never an execution boundary (§ 1.1 of `AGENTS.md`). Do not read a
> monthly table as a verdict.

## What code do I have to look at?

| # | file | what it is |
|---|---|---|
| 1 | `bt/optimal_config.py` | **the single source of truth** — every canonical knob + the 7 builders (`build_engine_kwargs` …) |
| 2 | `bt/vp_engine.py` | the forex wrapper — `qty_oz = lots × contract_size`, strips the forex fee knobs |
| 3 | `bt/_engine/vp_multi_target_engine.py` | **the engine** — bar loop, SL/TP geometry, trailing stop, fill models |
| 4 | `bt/_engine/ict_signals.py` | the FVG detector — produces the zone set |
| 5 | `bt/_engine/volume_profile.py` | the VP profile — produces the HVN shelves (the TP targets) |
| 6 | `bt/_engine/swing_cap.py` | NB72 daily swing cap — the TP ceiling |
| 7 | `bt/_engine/rolling_p95.py` | NB91 rolling p95 cap — the second TP ceiling |
| 8 | `bt/_engine/market_structure.py` | BoS/CHoCH — feeds the 5-event sequence whitelist |
| 9 | `bt/_engine/vol_state.py` | the measures the two vol gates read |
| 10 | `bt/_engine/timebase.py` | `to_ns_arr` — **nanoseconds**, the repo's one timebase rule |
| 11 | `runtime/sweeps/cached_sweep.py` | `load_bars`, `ArtifactCache`, `compute_metrics` — the reusable plumbing |
| 12 | `runtime/diagnostics/leg_ledger.py` | the canonical ledger — fingerprint, warmup prefix, per-leg records |
| 13 | `runtime/diagnostics/leg_buckets.py` | the four buckets + drift/excess + payoff geometry |
| 14 | `runtime/diagnostics/daily_direction_dependence.py` | the **daily** direction regression (n ≈ 577 days) |
| 15 | `runtime/results/registry.py` | the durable, commit-stamped results registry |
| 16 | `AGENTS.md`, `docs/evaluation.md` | the operating rules and the evidence |

**Cost warning.** The full corpus (2024-01-01 → 2026-07-16) is 41.2 M bars and holds
~6 GB. The artifact build is ~45 s + ~30 s and is *cached on disk*; the engine run is
~5 min per arm. **Never run two heavy cells concurrently** (§ 7) — memory contention
causes a ~9× slowdown, and it makes the test suite look red for no reason.

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# SETUP — imports, repo root, paths
# ─────────────────────────────────────────────────────────────────────
import copy
import json
import sys
import time
from pathlib import Path

REPO = Path(r'C:\coding\sniper_mt_vp_forex_bt_only')
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import numpy as np
import pandas as pd

from bt.optimal_config import (
    FOREX_LOTS, FOREX_CONTRACT_SIZE, FOREX_LEVERAGE,
    FOREX_COMMISSION_PER_LOT_USD, FOREX_SPREAD_PER_OZ_USD,
    ENTRY_MODE, ENTRY_FILL_MODEL, FILL_MODEL,
    SL_MODE, SL_ZONE_MULT, SL_BUFFER_USD, SL_NTH_HVN_N, SL_MIN_USD, SL_MAX_USD,
    TP_MODE, NTH_HVN_N, TP_ZONE_MULT_FALLBACK, TP_HVN_PROXY_FRAC,
    TRAIL_MODE, TRAIL_ATR_MULT, TRAIL_ACTIVATION_MULT, TRAIL_GIVEBACK_FRAC, TRAIL_SIDES,
    VOL_STATE_MIN_RANGE_NORM, VOL_STATE_MIN_EXPAND_PREV_DAY, VOL_STATE_MAX_EXPAND_PREV_DAY,
    STRUCTURE_SEQ_WHITELIST, TRADE_SIDES, MAX_SAME_DIR_OPEN, CONFLICT_POLICY,
    CONTRA_CLOSE, CONTRA_CLOSE_SPREAD_USD,
    SWING_CAP_PCT, TP_P95_MULT,
    DETECTOR_FVG_MIN_ZONE_USD, DETECTOR_RESAMPLE_SECS,
    STRUCTURE_PIVOT_LEN, STRUCTURE_RESAMPLE_SECS,
    build_engine_kwargs, _derive_taker_fee_bps,
)
from bt.vp_engine import run_vp_sniper_backtest_forex
from bt._engine.timebase import to_ns_arr
from runtime.sweeps.cached_sweep import (
    ArtifactCache, load_bars, compute_metrics, DEFAULT_DATA,
)
from runtime.results import registry

DATA = Path(DEFAULT_DATA)
REPORTS = REPO / 'runtime' / 'diagnostics' / '_reports'
REPORTS.mkdir(parents=True, exist_ok=True)

print('repo :', REPO)
print('data :', DATA, '| exists =', DATA.exists())

## §0 — The canonical recipe, read from the one source of truth

Nothing here is hardcoded in the notebook. `bt/optimal_config.py` is the ONLY place a
canonical value lives, and `build_engine_kwargs()` is the ONLY sanctioned way to turn it
into engine kwargs. An unknown override key raises `KeyError` — a typo can never
silently become a no-op.

In [ ]:
def show_recipe():
    qty = FOREX_LOTS * FOREX_CONTRACT_SIZE
    spot = 3000.0
    bps = _derive_taker_fee_bps(
        spot=spot,
        commission_per_lot_usd=FOREX_COMMISSION_PER_LOT_USD,
        spread_per_oz_usd=FOREX_SPREAD_PER_OZ_USD,
        lots=FOREX_LOTS, contract_size=FOREX_CONTRACT_SIZE)
    # engine fee = (exit+entry) * qty * bps/10000  ~=  2 * spot * qty * bps/10000
    rt = 2.0 * spot * qty * bps / 10000.0
    rows = [
        ('SIZING', f'{FOREX_LOTS} lots x {FOREX_CONTRACT_SIZE} contract = {qty:.2f} oz @ {FOREX_LEVERAGE:.0f}x'),
        ('FEE', f'${FOREX_COMMISSION_PER_LOT_USD}/lot + ${FOREX_SPREAD_PER_OZ_USD}/oz -> ${rt:.4f}/round-trip at ${spot:.0f}'),
        ('ENTRY', f'mode={ENTRY_MODE} fill={ENTRY_FILL_MODEL} exit_fill={FILL_MODEL}'),
        ('SL', f'mode={SL_MODE} zone_mult={SL_ZONE_MULT} nth_hvn={SL_NTH_HVN_N} floor=${SL_MIN_USD} ceiling=${SL_MAX_USD}'),
        ('TP', f'mode={TP_MODE} nth_hvn={NTH_HVN_N} fallback_mult={TP_ZONE_MULT_FALLBACK} proxy_frac={TP_HVN_PROXY_FRAC}'),
        ('TRAIL', f'mode={TRAIL_MODE} k={TRAIL_ATR_MULT} act={TRAIL_ACTIVATION_MULT} giveback={TRAIL_GIVEBACK_FRAC} sides={TRAIL_SIDES!r}'),
        ('VOL GATE', f'range_norm>={VOL_STATE_MIN_RANGE_NORM}  expand_prev_day>={VOL_STATE_MIN_EXPAND_PREV_DAY}  max_expand={VOL_STATE_MAX_EXPAND_PREV_DAY}'),
        ('STRUCTURE', f'pivot_len={STRUCTURE_PIVOT_LEN} @{STRUCTURE_RESAMPLE_SECS}s  whitelist={len(STRUCTURE_SEQ_WHITELIST)} keys'),
        ('BOOK', f'sides={TRADE_SIDES!r} max_same_dir={MAX_SAME_DIR_OPEN} conflict={CONFLICT_POLICY!r} contra_close={CONTRA_CLOSE}@${CONTRA_CLOSE_SPREAD_USD}'),
        ('CAPS', f'swing_pct={SWING_CAP_PCT}  p95_mult={TP_P95_MULT}'),
        ('DETECTOR', f'fvg_min=${DETECTOR_FVG_MIN_ZONE_USD} cadence={DETECTOR_RESAMPLE_SECS}s'),
    ]
    for k, v in rows:
        print(f'{k:<10} {v}')
    print()
    print(f'WHITELIST ({len(STRUCTURE_SEQ_WHITELIST)} keys) — a key PRESENT means TAKE:')
    for k in sorted(STRUCTURE_SEQ_WHITELIST):
        print('   ', k)


show_recipe()

## §1 — Load the bars

The canonical parquet stores `time` as `timestamp[ms, tz=UTC]`. **Every timestamp that
crosses a module boundary is int64 epoch NANOSECONDS** — use `to_ns_arr(bars)` and
never `.astype('int64')` on the column (that yields *milliseconds* and has silently
cost this repo a full run before). The assertion below is the guard.

In [ ]:
START, END = '2024-01-01', '2026-07-16'
WARMUP_DAYS = 14
EXEC_START = (pd.Timestamp(START, tz='UTC')
              - pd.Timedelta(days=WARMUP_DAYS)).strftime('%Y-%m-%d')

bars = load_bars(DATA, EXEC_START, END)
print('bars  :', f'{len(bars):,}')
print('cols  :', list(bars.columns))
print('window:', bars['time'].iloc[0], '->', bars['time'].iloc[-1])
print('close : min %.2f  median %.2f  max %.2f' % (
    bars['close'].min(), bars['close'].median(), bars['close'].max()))

times_ns = to_ns_arr(bars)
print('ns[0] :', int(times_ns[0]), '(must be ~1e18 = nanoseconds, NOT 1e12 = ms)')
assert 1e18 <= abs(int(times_ns[0])) < 1e19, 'timestamp unit error: ms/ns mix-up'

# The spot used to derive taker_fee_bps. It must be the REAL median of this
# file: holding it at the $3,000 default while the corpus trades at $4,060
# under-charges every trade by ~26% and flatters the book.
SPOT = float(np.median(bars['close'].to_numpy(dtype=np.float64)))
print('spot  :', SPOT)

## §2 — Build the artifacts (zones / VP profile / caps / structure / vol_state)

These are the expensive, **file-level** artifacts. `ArtifactCache` pickles them under
`notebooks/.cache/` keyed by a fingerprint of (data file, window, detector/VP/structure
shape, and the AST-hash of every artifact-producing source file). A cache HIT is
seconds; a MISS is ~45 s (zones) + ~30 s (VP) plus the caps.

**Do not bypass the cache.** The repo has been bitten twice by a stale artifact: the
fingerprint omitted `vol_state.py` once, which made the *same nominal config* produce
64,978 legs on one ledger and 37,159 on another.

In [ ]:
SCOPE = f'{DATA.stem}__{START.replace("-", "")}_{END.replace("-", "")}'

t0 = time.perf_counter()
art = ArtifactCache(DATA, SCOPE, EXEC_START, END).build_or_load(bars)
print('artifact cache:', 'HIT' if art.get('_cache_hit') else 'MISS',
      f'{time.perf_counter() - t0:.1f}s')
print('zones         :', f"{art['n_zones']:,}")
for k, v in art['build_secs'].items():
    print(f'   {k:<10} {v:7.1f}s')

print('artifact keys :', sorted(k for k in art if not k.startswith('_')))

## §3 — The engine call, wrapped once

`run_book()` below is the whole pipeline in one function, mirroring
`runtime/diagnostics/leg_ledger.run_window` exactly:

1. **warmup prefix** — the execution window starts `warmup_days` earlier; the prefix's
   legs are **discarded**, only its warm caps are kept. 14 days exceeds the longest
   history-dependent lookback (p95 cap 7d, swing cap 24h, VP 7d rolling).
2. **continuous execution** — one run over one window. Month boundaries are never
   execution boundaries.
3. **legs are keyed by the SIGNAL bar** — the causal decision point. `entry_bar` is the
   *fill* bar and is later (median 42 bars).
4. **per-leg drift / excess** — the exact algebraic decomposition
   `drift = (close[exit_bar] - close[entry_bar]) · side · qty`, `excess = gross - drift`.

The helper returns the legs in the **exact columnar schema the canonical ledger uses**,
so `runtime/diagnostics/leg_buckets.py` can analyse an in-notebook book unchanged.

In [ ]:
from runtime.diagnostics.leg_ledger import (
    EXIT_REASONS, SOURCES, _encode, LEDGER_PATH,
    build_ledger, load_ledger, ledger_arrays,
)

_EXIT_CODE = {r: i for i, r in enumerate(EXIT_REASONS)}
_SOURCE_CODE = {s: i for i, s in enumerate(SOURCES)}
LOCAL_CONTEXT_SECS = 3600

_BARS_MEMO = {}
_ART_MEMO = {}


def _legs_from_trades(trades, bars):
    """ClosedTrade list -> the canonical ledger's columnar schema."""
    n = len(trades)
    close = bars['close'].to_numpy(dtype=np.float64)
    times_ns = to_ns_arr(bars)
    nb = int(close.size)

    side = np.zeros(n, dtype=np.int8)
    outcome = np.zeros(n, dtype=np.int8)
    gross = np.zeros(n); net = np.zeros(n); fee = np.zeros(n)
    entry_price = np.zeros(n); exit_price = np.zeros(n)
    qty = np.zeros(n); sl_dist = np.zeros(n); tp_dist = np.zeros(n)
    hold = np.zeros(n)
    entry_bar = np.zeros(n, dtype=np.int64)
    exit_bar = np.zeros(n, dtype=np.int64)
    sig_bar = np.zeros(n, dtype=np.int64)
    entry_t = np.zeros(n, dtype=np.int64)
    exit_t = np.zeros(n, dtype=np.int64)
    exit_code = np.zeros(n, dtype=np.int8)
    source_code = np.zeros(n, dtype=np.int8)

    for i, t in enumerate(trades):
        d = 1 if int(t.direction) > 0 else -1
        g = float(t.pnl_gross_usd)
        side[i] = d
        outcome[i] = 1 if g > 0 else -1
        gross[i] = g
        net[i] = float(t.pnl_net_usd)
        fee[i] = float(getattr(t, 'fee_usd', 0.0))
        entry_price[i] = float(t.entry_price)
        exit_price[i] = float(t.exit_price)
        qty[i] = float(getattr(t, 'qty_oz', 0.0))
        sl_dist[i] = abs(float(getattr(t, 'initial_sl_distance_usd', 0.0)
                              or getattr(t, 'sl_distance_usd', 0.0) or 0.0))
        tp_dist[i] = abs(float(getattr(t, 'initial_tp_distance_usd', 0.0)
                              or getattr(t, 'tp_distance_usd', 0.0) or 0.0))
        hold[i] = float(getattr(t, 'hold_secs', 0.0))
        entry_bar[i] = int(getattr(t, 'entry_bar', 0))
        exit_bar[i] = int(getattr(t, 'exit_bar', 0))
        sig_bar[i] = int(getattr(t, 'signal_bar', 0))
        entry_t[i] = int(t.entry_time_ns)
        exit_t[i] = int(t.exit_time_ns)
        exit_code[i] = _EXIT_CODE.get(str(getattr(t, 'exit_reason', 'other')),
                                    len(EXIT_REASONS) - 1)
        source_code[i] = _SOURCE_CODE.get(str(getattr(t, 'source', 'other')),
                                      len(SOURCES) - 1)

    e_idx = np.clip(entry_bar, 0, max(nb - 1, 0))
    x_idx = np.clip(exit_bar, 0, max(nb - 1, 0))
    s_idx = np.clip(sig_bar, 0, max(nb - 1, 0))

    # The REAL counterfactual: a passive hold of the same side over the same
    # window. qty is PER LEG (a TP1 remainder leg is half size).
    drift = (close[x_idx] - close[e_idx]) * side.astype(float) * qty
    excess = gross - drift

    # The market's drift over the hour BEFORE the signal bar — the per-leg
    # replacement for the monthly UP/DOWN/FLAT label.
    target = times_ns[s_idx] - int(LOCAL_CONTEXT_SECS * 1_000_000_000)
    pos = np.clip(np.searchsorted(times_ns, target, side='left'), 0, max(nb - 1, 0))
    local = close[s_idx] - close[pos]

    return {
        'side': side, 'outcome': outcome,
        'gross': gross, 'net': net, 'fee': fee,
        'move_usd': np.abs(exit_price - entry_price) * qty,
        'qty': qty,
        'drift_usd': drift, 'excess_usd': excess,
        'flip_gross': -gross,   # TAUTOLOGY (-gross). Documented trap, never a verdict.
        'local_drift_1h': local,
        'entry_price': entry_price, 'exit_price': exit_price,
        'sl_dist': sl_dist, 'tp_dist': tp_dist, 'hold_secs': hold,
        'signal_time_ns': times_ns[s_idx] if nb else np.zeros(n, dtype=np.int64),
        'entry_time_ns': entry_t, 'exit_time_ns': exit_t,
        'entry_bar': entry_bar, 'exit_bar': exit_bar, 'signal_bar': sig_bar,
        'exit_code': exit_code, 'source_code': source_code,
    }


def _payload_from_legs(legs):
    """Wrap the leg columns into a ledger-shaped payload (exact int64 for ns)."""
    return {
        'legs': _encode(legs),
        'exit_reasons': list(EXIT_REASONS),
        'sources': list(SOURCES),
        'n_legs': int(len(legs['gross'])),
    }


def run_book(*, start=START, end=END, overrides=None, warmup_days=WARMUP_DAYS,
             data=DATA, label='book'):
    """One continuous execution over one window. Returns scored legs + metrics."""
    ov = dict(overrides or {})
    exec_start = (pd.Timestamp(start, tz='UTC')
                  - pd.Timedelta(days=warmup_days)).strftime('%Y-%m-%d')
    scope = f'{data.stem}__{start.replace("-", "")}_{end.replace("-", "")}'

    bkey = (str(data), exec_start, end)
    if bkey not in _BARS_MEMO:
        _BARS_MEMO[bkey] = load_bars(data, exec_start, end)
    b = _BARS_MEMO[bkey]
    if b.empty:
        raise RuntimeError(f'no bars for {label}')

    akey = (str(data), scope, exec_start, end)
    if akey not in _ART_MEMO:
        _ART_MEMO[akey] = ArtifactCache(data, scope, exec_start, end).build_or_load(b)
    a = _ART_MEMO[akey]

    spot = float(np.median(b['close'].to_numpy(dtype=np.float64)))
    ek = build_engine_kwargs(
        spot=spot,
        swing_cap=a['swing_cap'], p95_cap=a['p95_cap'],
        structure=a['structure'], vol_state=a['vol_state'],
        **ov)

    # The per-bar equity curve is 330 MB on the 1y file. Mirror the sweep
    # driver: record it only when the bar count is small enough.
    record_curve = len(b) <= 8_000_000

    t0 = time.perf_counter()
    res, _diag = run_vp_sniper_backtest_forex(
        bars=b, zones=a['zones'], vp_set=a['vp_set'],
        lots=FOREX_LOTS, contract_size=FOREX_CONTRACT_SIZE,
        engine_kwargs=ek, record_equity_curve=record_curve)
    wall = time.perf_counter() - t0

    legs = _legs_from_trades(res.trades, b)

    # Keep only the legs whose SIGNAL time falls in the scored window.
    lo = int(pd.Timestamp(start, tz='UTC').value)
    hi = int(pd.Timestamp(end, tz='UTC').value)
    st = legs['signal_time_ns']
    keep = (st >= lo) & (st < hi)
    legs = {k: v[keep] for k, v in legs.items()}

    # Shallow copy so compute_metrics sees the SCORED legs only.
    scored = copy.copy(res)
    scored.trades = [t for t, k in zip(res.trades, keep) if k]
    metrics = compute_metrics(scored)

    print(f'[{label}] legs={len(legs["gross"]):,}  engine={wall:.1f}s  '
          f'art={"HIT" if a.get("_cache_hit") else "MISS"}')
    return {'label': label, 'legs': legs, 'metrics': metrics, 'wall': wall,
            'overrides': ov, 'result': res, 'bars': b, 'artifacts': a}


print('helpers ready')

## §4 — Run the canonical book

No overrides = the shipped recipe. Expect ~7,122 legs, gross ≈ +$5,623, net ≈ +$2,298
on the full corpus (`trail_sides='short'`, promoted 2026-11-04).

In [ ]:
canonical = run_book(label='canonical')
m = canonical['metrics']

print('=' * 70)
print('CANONICAL BOOK  (criterion = NET PROFIT)')
print('=' * 70)
print(f"legs          {m['n_trades']:,}  ({m['n_long']:,} long / {m['n_short']:,} short)")
print(f"trades/day    {m['trades_per_day']:.2f}   (target band 30-70, WAIVED for this recipe)")
print(f"hit rate      {m['hit_rate']:.2%}")
print(f"gross         ${m['total_pnl_gross_usd']:,.2f}")
print(f"fees          ${m['total_fees_usd']:,.2f}  (${m['total_fees_usd']/max(m['n_trades'],1):.4f}/leg measured)")
print(f"NET           ${m['total_pnl_net_usd']:,.2f}")
print(f"profit factor {m['profit_factor'] if m['profit_factor'] is None else round(m['profit_factor'], 3)}")
print(f"payoff ratio  {m['payoff_ratio'] if m['payoff_ratio'] is None else round(m['payoff_ratio'], 3)}")
print(f"max drawdown  ${m['max_drawdown_usd']:,.2f}  (source: {m['max_dd_source']})")
print(f"median hold   {m['median_hold_secs']/3600:.2f} h")
print(f"median SL/TP  ${m['median_sl_distance_usd']:.2f} / ${m['median_tp_distance_usd']:.2f}")
print()
print('exit reasons  ', m['exit_reason_breakdown'])
print('sources       ', m['source_breakdown'])
print()
print('gate counters:')
for k in ('n_signals_emitted', 'n_immediate_filled', 'n_sniper_filled',
          'n_dropped_vol_state', 'n_dropped_structure', 'n_dropped_policy',
          'n_dropped_seq_whitelist', 'n_orders_placed', 'n_orders_filled'):
    v = getattr(canonical['result'], k, None)
    if v is not None:
        print(f'   {k:<26} {v:,}')

## §5 — The leg-level evaluation (the sanctioned unit)

This is `runtime/diagnostics/leg_buckets.analyse()` running on the in-notebook book.
It answers the three questions:

1. **Is there an edge at all?** — the per-leg gross EV against the **measured** round trip.
   A gross edge below the fee line is *not* an edge.
2. **Is the edge direction-independent?** — the `drift` / `excess` decomposition.
   `excess` is what the entries added on top of a passive hold.
3. **Is the loss concentrated?** — the loss buckets' mean and tail.

> **Read the `excess` caveat.** `excess` benchmarks the realised P&L against a
> *close-to-close* passive hold, but the engine books P&L at the **actual SL/TP fill
> price**. For a stop or target bucket, `excess` is therefore the **execution-gap
> differential**, not a timing measure — its sign flips with book composition. The
> whole-book and per-side forms DO hold.

In [ ]:
from runtime.diagnostics import leg_buckets

canonical_payload = _payload_from_legs(canonical['legs'])
analysis = leg_buckets.analyse(canonical_payload)
leg_buckets.print_report(canonical_payload, analysis)
print()
print('VERDICT:', analysis['verdict'])
print()
print('--- direction dependence (drift = market beta, excess = timing) ---')
d = analysis['directional']
for k, v in d.items():
    if isinstance(v, dict):
        print(f"   {k:<6} n={v['n']:>6}  gross ${v['gross']:>10,.2f}  "
              f"drift ${v['drift']:>10,.2f}  excess ${v['excess']:>10,.2f}  "
              f"t_excess={v['t_excess']:+.2f}")
    elif isinstance(v, (int, float)):
        print(f'   {k:<34} {v:,.4f}')

## §6 — The payoff geometry: signal problem or fee problem?

`edge_pp = 100 · (hit_rate − p_breakeven)`.

* `edge_pp ≤ 0` → **no signal** (the side loses before fees).
* `edge_pp > 0` but `gross_per_leg < fee_per_leg` → **fee-limited** (the signal is real
  but too small to pay for itself).
* `gross_per_leg ≥ fee_per_leg` → **profitable**.

These two failure modes need **different** fixes, and the old monthly paradigm could
not tell them apart.

In [ ]:
pg = analysis['payoff_geometry']
if isinstance(pg, dict):
    print(f"{'side':<8}{'n':>7}{'avg_win':>10}{'avg_loss':>10}{'payoff':>9}"
          f"{'p_be':>8}{'hit':>8}{'edge_pp':>9}  diagnosis")
    for side_name in ('long', 'short'):
        st = pg.get(side_name)
        if not st or not st.get('n'):
            continue
        print(f"{side_name:<8}{st['n']:>7}{st.get('avg_win', 0):>10.4f}"
              f"{st.get('avg_loss', 0):>10.4f}{st.get('payoff_ratio', 0):>9.3f}"
              f"{st.get('breakeven_win_rate', 0):>8.2%}{st.get('hit_rate', 0):>8.2%}"
              f"{st.get('edge_pp', 0):>9.2f}  {st.get('diagnosis', '')}")
else:
    print(pg)

print()
print('--- exit-reason mix (the trail scopes the SHORT side only) ---')
for r, st in sorted(analysis['exit_breakdown'].items(),
                    key=lambda kv: -kv[1]['n']):
    print(f"   {r:<26} {st['n']:>7,}  gross ${st['gross']:>12,.2f}")

## §7 — The canonical ledger route (reuse the committed measurement)

The ledger is the repo's durable artifact: a fingerprinted, cached, per-leg record with
the same schema this notebook just built. Reading it is **cheaper and safer** than
re-running, and it is what every downstream consumer (`leg_buckets`,
`leg_random_entry`, the placebo harness, the test suite) reads.

> **⚠️ DO NOT overwrite the canonical ledger with a different window.**
> `build_ledger()` refuses to write a *variant recipe* to the canonical path, but it does
> **not** refuse a different **window**. Writing a 1-month ledger to `LEDGER_PATH`
> silently replaces the 927-day canonical artifact and every consumer then analyses a
> different book while believing it is reading the canonical one. Always pass an
> explicit `out_path` for anything that is not the canonical corpus window.

This is the single most important trap in the whole harness.

In [ ]:
# The canonical corpus window -> the canonical artifact. Read it if it exists.
print('canonical ledger path:', LEDGER_PATH)
print('exists            :', LEDGER_PATH.exists())
print()

if LEDGER_PATH.exists():
    led = load_ledger(LEDGER_PATH, expect_canonical=True)
    arr = ledger_arrays(led)
    n = len(arr['gross'])
    g = float(arr['gross'].sum()); f = float(arr['fee'].sum())
    print(f"ledger legs {n:,}  gross ${g:,.2f}  fees ${f:,.2f}  net ${g - f:,.2f}")
    print(f"gross/leg ${g/n:+.4f}   fee/leg ${f/n:.4f}   gross/fee {(g/n)/(f/n):.3f}x")
    print(f"drift  ${float(arr['drift_usd'].sum()):,.2f}   "
          f"excess ${float(arr['excess_usd'].sum()):,.2f}")
    print(f"built  {led.get('built_utc')}   fp {led.get('fingerprint')}")
else:
    print('no canonical ledger on disk — build it with:')
    print('    python -m runtime.diagnostics.leg_ledger')


# Building a ledger for a NON-canonical window: always give it its own path.
VARIANT_PATH = REPORTS / f'leg_ledger_nb_{START}_{END}.json'
print()
print('variant path (never the canonical one):', VARIANT_PATH)

## §8 — The DAILY direction-dependence test

The sanctioned instrument (§ 1.1 rule 7). A **monthly** regression has n = 13..31
points, so its correlation's standard error is ~0.32 in Fisher-z space — every monthly
`r` this repo ever quoted sat inside its own noise band. The **daily** aggregation
has n ≈ 577 days and is decisive.

The per-leg decomposition (the primary number) uses n = thousands of legs.

In [ ]:
from runtime.diagnostics import daily_direction_dependence as ddd

mkt = ddd._market_daily(START, END)
mkt_map = dict(zip(mkt['day'], mkt['mkt_ret_pct']))
book = ddd.analyse_book(canonical_payload, mkt_map)

print('DAILY direction dependence')
print('=' * 70)
print(f"legs                {book['n_legs']:,}")
print(f"days                {book['n_days']:,}")
reg = book['daily_regression']
print(f"daily corr          {reg.get('corr', 0):+.3f}   Fisher-z t = {reg.get('corr_t', 0):+.2f}")
print(f"r^2                 {reg.get('r2', 0):.3f}")
print(f"beta $/1%           {reg.get('beta_usd_per_pct', 0):+.2f}")
print(f"verdict             {book['verdict']}")
print()
for scope in ('per_leg', 'per_leg_long', 'per_leg_short'):
    b = book.get(scope) or {}
    if not b.get('legs'):
        continue
    print(f"{scope:<16} n={b['legs']:>6}  gross/leg {b['gross_per_leg']:+.4f}  "
          f"drift/leg {b['drift_per_leg']:+.4f} (t={b['drift_t']:+.2f})  "
          f"excess/leg {b['excess_per_leg']:+.4f} (t={b['excess_t']:+.2f})")

## §9 — A/B a knob, end to end

> **§ 3.11 — a drop-set or whitelist MUST be measured END TO END by re-running the
> engine.** A post-hoc leg filter is a *screen* for which hypotheses deserve a re-run,
> never the verdict itself. `contra_close` nets opposing legs **pairwise**, so removing
> legs removes the netting's counterparties — a post-hoc prediction cannot see that.

Each arm re-runs the engine over the **same** window. The artifacts are shared (they do
not depend on the arm), so only the engine loop is repeated.

The arms below are the `trail_sides` scope grid — the promotion that shipped on
2026-11-04. **The criterion is NET PROFIT** (`excess` is a diagnostic, never a gate).

In [ ]:
ARMS = [
    ('trail OFF (baseline)', {'trail_mode': 'off'}),
    ('trail | both',         {'trail_sides': 'both'}),
    ('trail | long',         {'trail_sides': 'long'}),
    ('trail | short (SHIPPED)', {'trail_sides': 'short'}),
]

rows = []
for name, ov in ARMS:
    out = run_book(overrides=ov, label=name)
    L = out['legs']
    n = int(len(L['gross']))
    g = float(L['gross'].sum()); f = float(L['fee'].sum())
    rows.append({
        'arm': name,
        'legs': n,
        'gross': g,
        'fees': f,
        'net': g - f,
        'gross/leg': g / n if n else 0.0,
        'fee/leg': f / n if n else 0.0,
        'gross/fee': (g / n) / (f / n) if n and f else 0.0,
        'long net': float(L['net'][L['side'] > 0].sum()),
        'short net': float(L['net'][L['side'] < 0].sum()),
        'trail exits': int((L['exit_code'] == EXIT_REASONS.index('trail_stop')).sum()),
        'secs': out['wall'],
    })

ab = pd.DataFrame(rows)
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 50)
print()
print('=' * 78)
print('SCOPE GRID  (criterion = NET PROFIT)')
print('=' * 78)
print(ab.to_string(index=False, float_format=lambda v: f'{v:,.4f}'))

## §10 — Record the measurement in the results registry

> **A number nobody can find is not evidence.** Every run that produces a number is
> recorded in `results/` with a commit stamp.

`config_fp` identifies the **parameters only** — the commit is deliberately excluded,
so the same parameters re-measured after a bug fix produce two records with an
identical `config_fp` and a different `commit`, and the diff between them *is* the
effect of the fix.

In [ ]:
L = canonical['legs']
n = int(len(L['gross']))
g = float(L['gross'].sum()); f = float(L['fee'].sum())

entry = registry.record_run(
    kind='manual_backtest',
    params={
        'start': START,
        'end': END,
        'warmup_days': WARMUP_DAYS,
        'overrides': {k: repr(v) for k, v in sorted({}.items())},
        'data': str(DATA),
        'lots': FOREX_LOTS,
        'contract_size': FOREX_CONTRACT_SIZE,
    },
    summary={
        'legs': n,
        'gross': round(g, 6),
        'fees': round(f, 6),
        'net': round(g - f, 6),
        'gross_per_leg': round(g / n, 6) if n else None,
        'fee_per_leg': round(f / n, 6) if n else None,
        'trades_per_day': round(n / max((pd.Timestamp(END, tz='UTC') -
                                          pd.Timestamp(START, tz='UTC')).days, 1), 4),
    },
    payload={
        'metrics': canonical['metrics'],
        'analysis_verdict': analysis['verdict'],
    },
)
print('recorded run_id :', entry.run_id)
print('config_fp       :', entry.config_fp)
print('commit          :', entry.commit, '| dirty =', entry.dirty)
print()
print('read it back:')
print('    python -m runtime.results.cli show', entry.run_id)
print('    python -m runtime.results.cli list --kind manual_backtest')

## §11 — Plots (optional)

The equity curve is only recorded on short windows (330 MB on 1y). On the corpus the
fallback is the closed-trade cumulative P&L, which lags the drawdown by `hold_secs` per
trade.

In [ ]:
import matplotlib
matplotlib.use('module://matplotlib_inline.backend_inline')
import matplotlib.pyplot as plt

L = canonical['legs']
order = np.argsort(L['exit_time_ns'])
exit_ts = pd.to_datetime(L['exit_time_ns'][order], unit='ns', utc=True)
cum = np.cumsum(L['net'][order])

fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True,
                         gridspec_kw={'height_ratios': [2, 1]})

axes[0].plot(exit_ts, cum, lw=1.0, color='#2b6cb0')
axes[0].axhline(0, color='black', lw=0.6, alpha=0.4)
axes[0].set_title('Cumulative NET P&L (closed trades, by exit time)')
axes[0].set_ylabel('USD')
axes[0].grid(alpha=0.25)

daily = pd.Series(L['net'], index=pd.to_datetime(L['signal_time_ns'], unit='ns', utc=True)).resample('D').sum()
axes[1].bar(daily.index, daily.values, width=1.0, color=np.where(daily.values >= 0, '#2f855a', '#c53030'))
axes[1].axhline(0, color='black', lw=0.6)
axes[1].set_title('Daily NET P&L (bucketed by the SIGNAL day — reporting only)')
axes[1].set_ylabel('USD/day')
axes[1].grid(alpha=0.25)

plt.tight_layout()
plt.show()

## §12 — The traps, in one place

1. **Never overwrite the canonical ledger with a different window.** `build_ledger()`
   guards the *recipe*, not the *window*. Pass an explicit `out_path`.
2. **Timestamps are nanoseconds.** Use `to_ns_arr(bars)`; `astype('int64')` gives
   milliseconds and has silently voided a full corpus run before.
3. **The spot must be the file's real median.** `build_engine_kwargs(spot=...)` derives
   `taker_fee_bps` from it. Holding `$3,000` while the file trades at `$4,060`
   under-charges every trade by ~26%.
4. **The fee is a MEASUREMENT, not the `$0.47` spec.** The engine's fee scales with the
   fill price; the corpus measures ≈ `$0.50`/leg. Verdict against the measured fee.
5. **The month is a reporting dimension, never an evaluation boundary.** n = 13..31
   monthly points cannot decide anything.
6. **`excess` is not a promotion gate.** The promotion criterion is **NET PROFIT**.
7. **Re-measure a volume gate whenever the exit geometry changes.** `sl_max_usd` and
   `vol_state_min_range_norm` are **paired**, not independent.
8. **Never run two heavy cells concurrently** — a ~9× slowdown, and it makes the test
   suite report failures that are not code defects.
9. **`sl_max_usd = 15.00` is UNMEASURED** (it was an assumption, not a sweep result),
   and 70.88% of legs are capped there. The corpus re-measurement of the paired
   `range_norm × sl_max_usd` grid is still **owed**.
10. **Do not run the zone-shuffle placebo unprompted.** It costs ~10 min and **rewrites**
    `zone_shuffle_placebo.json`, which the guard tests read. Ask first.